# CMIP masked-point prediction — baseline simple

Simple predictor baseline for the masked-point prediction task.

This notebook uses rigorous preprocessing:
1. train/validation/test split is created before standardization;
2. normalization statistics are fitted only on training samples;
3. masked points are excluded from the normalization fit;
4. hidden point values are replaced by zero after standardization;
5. an explicit binary mask channel indicates which geographic point is hidden;
6. the predictor is trained only to predict the masked point values.


### Imports

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### Hyperparameters

In [ ]:
setup_name = "baseline_simple"

In [ ]:
num_sample = 2500000
mask_strategy = "central_1"  # options: central_1, central_6, checkerboard, hidden_bottom, keep_central_6, keep_central_1
val_fraction = 0.05
test_fraction = 0.15

### Load precomputed CMIP samples and define the mask

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NG_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])
baseline_train_climates = ["historical"]

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch

# Define the masked geographic point(s).

prediction_task = "masked_point_reconstruction"
masked_point_indices = None  # if None, it will be set to the central point after loading grid metadata
mask_fill_value = 0.0

if masked_point_indices is None:
    if mask_strategy == "central_1":
        masked_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)

    elif mask_strategy == "central_6":
        center_i = n_lat // 2
        center_j = n_lon // 2
        masked_point_indices = np.array([
            (center_i - 1) * n_lon + (center_j - 1),
            center_i * n_lon + (center_j - 1),
            (center_i - 1) * n_lon + center_j,
            center_i * n_lon + center_j,
            (center_i - 1) * n_lon + (center_j + 1),
            center_i * n_lon + (center_j + 1),
        ], dtype=int)

    elif mask_strategy == "checkerboard":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if (i + j) % 2 == 0
        ], dtype=int)
    
    elif mask_strategy == "hidden_bottom":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if i < n_lat // 2
        ], dtype=int)

    elif mask_strategy == "keep_central_6":
        visible_point_indices = np.array([
            5 * n_lon + 2,
            5 * n_lon + 3,
            5 * n_lon + 4,
            4 * n_lon + 2,
            4 * n_lon + 3,
            4 * n_lon + 4,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)
    
    elif mask_strategy == "keep_central_1":
        visible_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)

    else:
        raise ValueError(f"Unknown mask_strategy={mask_strategy!r}.")
else:
    masked_point_indices = np.asarray(masked_point_indices, dtype=int)

if masked_point_indices.ndim != 1 or masked_point_indices.size == 0:
    raise ValueError("masked_point_indices must be a non-empty 1D array of grid-point indices.")
if masked_point_indices.min() < 0 or masked_point_indices.max() >= grid_points_per_patch:
    raise ValueError(
        f"masked_point_indices must be between 0 and {grid_points_per_patch - 1}; "
        f"got {masked_point_indices.tolist()}."
    )

# Flattening convention: [var0_point0, ..., var0_point69, var1_point0, ..., var15_point69].
masked_feature_columns = np.array(
    [var_idx * grid_points_per_patch + point_idx
     for var_idx in range(n_variables_full)
     for point_idx in masked_point_indices],
    dtype=int,
)
visible_feature_columns = np.setdiff1d(
    np.arange(expected_dim_full, dtype=int),
    masked_feature_columns,
    assume_unique=False,
)

masked_prediction_labels = [
    f"{var}@point_{point_idx}"
    for var in selected_variables_full
    for point_idx in masked_point_indices
]

prediction_task = "masked_point_reconstruction"
mask_channel_name = "mask_hidden_point"
selected_variables = list(selected_variables_full) + [mask_channel_name]
input_value_dim = expected_dim_full
input_mask_dim = grid_points_per_patch
input_dim_with_mask = input_value_dim + input_mask_dim

for c, X in features_by_climate_full.items():
    if X.shape[1] != expected_dim_full:
        raise ValueError(f"Unexpected feature dimension for {c}: got {X.shape[1]}, expected {expected_dim_full}.")

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Prediction task: {prediction_task}")
print(f"Input channels used by the model: {selected_variables}")
print(f"Masked grid point indices: {masked_point_indices.tolist()}")
print(f"Masked feature columns: {masked_feature_columns.tolist()}")
print(f"Prediction target dimension: {len(masked_feature_columns)} = {n_variables_full} variables x {len(masked_point_indices)} hidden point(s)")
print(f"Prediction labels: {masked_prediction_labels}")


### Split first, then rigorous standardization

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices separately within each climate."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        split_indices[climate] = {
            "train": indices[:n_train],
            "val": indices[n_train:n_train + n_val],
            "test": indices[n_train + n_val:],
        }

    return split_indices


# Split raw data before any normalization.
ae_split_indices = build_split_indices(
    features_by_climate_full,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimisation ───────
_eval_only_climates = [c for c in climate_order if c not in baseline_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c]      = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt : {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx
del _eval_only_climates
# ── End RAM optimisation ──────────────────────────────────────────────────────

visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)

# The scaler is fitted only on training samples from these climates.
# This preserves climate shift and avoids using validation/test information.
scaler_fit_climates = ["historical"]

scaler_fit_climates = [c for c in scaler_fit_climates if c in climate_order]
if not scaler_fit_climates:
    raise ValueError("scaler_fit_climates is empty after filtering against climate_order.")

# One mean/std per physical variable, estimated only from visible points and training samples.
variable_means = np.zeros(n_variables_full, dtype=np.float64)
variable_stds = np.ones(n_variables_full, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables_full):
    visible_cols_for_var = var_idx * grid_points_per_patch + visible_point_indices
    train_visible_values = []

    for climate in scaler_fit_climates:
        train_idx = ae_split_indices[climate]["train"]
        X_train_raw = np.asarray(features_by_climate_full[climate][train_idx])
        train_visible_values.append(X_train_raw[:, visible_cols_for_var].reshape(-1))

    train_visible_values = np.concatenate(train_visible_values)
    finite_values = train_visible_values[np.isfinite(train_visible_values)]

    if finite_values.size == 0:
        raise ValueError(f"No finite visible training values found for variable {var_name!r}.")

    if var_name == "pr":
        finite_values = np.log1p(finite_values * 86400)

    mu = float(np.mean(finite_values))
    sigma = float(np.std(finite_values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    variable_means[var_idx] = mu
    variable_stds[var_idx] = sigma

scaled_features_by_climate = {}
label_variable_by_climate = {}

mask_channel_template = np.zeros(grid_points_per_patch, dtype=np.float32)
mask_channel_template[masked_point_indices] = 1.0

for climate in climate_order:
    X_raw = np.asarray(features_by_climate_full[climate], dtype=np.float32)
    X_scaled_full = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables_full):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled_full[:, cols_for_var] = (
            values - variable_means[var_idx]
        ) / variable_stds[var_idx]

    # Target: true standardized values at the hidden point(s).
    y_masked = X_scaled_full[:, masked_feature_columns].copy()

    # Input: standardized variables with hidden point values replaced by zero.
    X_values_masked = X_scaled_full.copy()
    X_values_masked[:, masked_feature_columns] = mask_fill_value

    # Explicit mask channel: 1 at hidden point(s), 0 elsewhere.
    mask_channel = np.tile(mask_channel_template, (X_values_masked.shape[0], 1))
    X_model_input = np.concatenate([X_values_masked, mask_channel], axis=1).astype(np.float32)

    scaled_features_by_climate[climate] = X_model_input
    label_variable_by_climate[climate] = y_masked.astype(np.float32)


input_dim = input_dim_with_mask
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])
meta_all = pd.concat([metadata_by_climate[c] for c in climate_order], ignore_index=True)

print("Rigorous preprocessing complete.")
print(f"Scaler fitted on climates: {scaler_fit_climates}")
print(f"Scaler fitted on train split only and visible points only: {visible_point_indices.size} / {grid_points_per_patch} points")
print(f"Input dimension: {input_dim} = {input_value_dim} standardized values + {input_mask_dim} mask indicators")
print(f"Prediction output dimension: {label_variable.shape[1]}")
print(pd.DataFrame({"variable": selected_variables_full, "mean": variable_means, "std": variable_stds}))


In [ ]:
# RAM Reduction
del features_by_climate_full

### Model Hyperparameters

In [ ]:
# Baseline predictor training
baseline_batch_size = 256
baseline_n_epochs = 100
baseline_patience = 10
baseline_learning_rate = 1e-3
baseline_weight_decay = 1e-5
baseline_hidden_dim = 128
baseline_n_hidden_layers = 5
baseline_checkpoint_path = Path("/glade/u/home/tsalin/CMIP/model_evaluation/Baseline_simple/baseline_mask_checkpoint.pt")
baseline_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)

# Evaluation
eval_batch_size = 4096

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### DataLoader helpers

In [ ]:
def make_labeled_loader(data_by_climate, labels_by_climate, split_indices, climate, split, batch_size, shuffle, drop_last):
    idx = split_indices[climate][split]
    X = np.asarray(data_by_climate[climate][idx], dtype=np.float32)
    y = np.asarray(labels_by_climate[climate][idx], dtype=np.float32)
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


### Simple baseline predictor

In [ ]:
class BaselinePredictor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128, n_hidden_layers: int = 5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.1))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # For this simple baseline, use the penultimate hidden representation as a diagnostic latent.
        h = x
        for layer in list(self.net.children())[:-1]:
            h = layer(h)
        return h


### Training

In [ ]:
def train_baseline_masked_point_predictor(
    train_climates=None,
    n_epochs: int = 100,
    lr: float = 1e-3,
    batch_size: int = 256,
    weight_decay: float = 1e-5,
    hidden_dim: int = 128,
    n_hidden_layers: int = 5,
    patience: int = 10,
    checkpoint_path=None,
    checkpoint_freq: int = 1,
):
    """Train a simple MLP predictor directly from masked standardized input to hidden point values."""
    if train_climates is None:
        train_climates = ["historical"]
    if train_climates != ["historical"]:
        raise ValueError("This baseline expects train_climates=['historical']")

    input_dim_local = next(iter(scaled_features_by_climate.values())).shape[1]
    output_dim_pred = int(len(masked_feature_columns))

    model = BaselinePredictor(
        input_dim=input_dim_local,
        output_dim=output_dim_pred,
        hidden_dim=hidden_dim,
        n_hidden_layers=n_hidden_layers,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    mse_loss_fn = nn.MSELoss()

    train_loader = make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "train", batch_size, True, True
    )
    val_loader = make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "val", batch_size, False, False
    )

    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    best_state = None
    history = []

    start_epoch = 1
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        _ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(_ckpt["model_state"])
        optimizer.load_state_dict(_ckpt["optimizer_state"])
        best_val = _ckpt["best_val"]
        best_epoch = _ckpt["best_epoch"]
        patience_counter = _ckpt["patience_counter"]
        best_state = _ckpt["best_state"]
        history = _ckpt["history"]
        start_epoch = _ckpt["epoch"] + 1
        print(f"[Checkpoint] Resume from epoch {start_epoch} (best val={best_val:.6g})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            y_pred = model(xb)
            loss = mse_loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                y_pred = model(xb)
                loss = mse_loss_fn(y_pred, yb)
                val_losses.append(float(loss.detach().cpu().item()))

        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        val_loss = float(np.mean(val_losses)) if val_losses else np.nan

        history.append({
            "epoch": epoch,
            "train_pred_loss": train_loss,
            "train_total_loss": train_loss,
            "val_pred_loss": val_loss,
            "val_total_loss": val_loss,
        })

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_state": best_state,
                "history": history,
            }, checkpoint_path)

        if np.isfinite(val_loss) and val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}, best val={best_val:.6g}")
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_state": best_state,
                        "history": history,
                    }, checkpoint_path)
                break

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | train={train_loss:.6g} | val={val_loss:.6g}")

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    return model, history_df, best_epoch


baseline_predictor, baseline_history_df, baseline_best_epoch = train_baseline_masked_point_predictor(
    train_climates=baseline_train_climates,
    n_epochs=baseline_n_epochs,
    lr=baseline_learning_rate,
    batch_size=baseline_batch_size,
    weight_decay=baseline_weight_decay,
    hidden_dim=baseline_hidden_dim,
    n_hidden_layers=baseline_n_hidden_layers,
    patience=baseline_patience,
    checkpoint_path=baseline_checkpoint_path,
    checkpoint_freq=baseline_checkpoint_freq,
)

print(f"Best epoch: {baseline_best_epoch}")
display(baseline_history_df.tail())


### Evaluation and output DataFrames

In [ ]:
component_order = {"prediction": 0}


# ---------------------------------------------------------------------
# Mask / visible metadata
# ---------------------------------------------------------------------
visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)

visible_feature_columns = np.setdiff1d(
    np.arange(expected_dim_full, dtype=int),
    masked_feature_columns,
    assume_unique=False,
)


def denormalize_masked_targets(
    y_normalized: np.ndarray,
    variable_means: np.ndarray,
    variable_stds: np.ndarray,
    n_masked_points: int,
    n_variables: int,
) -> np.ndarray:
    y_denorm = np.asarray(y_normalized, dtype=np.float32).copy()

    for var_idx, var_name in enumerate(selected_variables_full):
        start_col = var_idx * n_masked_points
        end_col = (var_idx + 1) * n_masked_points
        values = y_denorm[:, start_col:end_col] * variable_stds[var_idx] + variable_means[var_idx]
        if var_name == "pr":
            values = np.expm1(values)
        y_denorm[:, start_col:end_col] = values

    return y_denorm


def _build_meta_df(climate, metadata, latent_dim):
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment", "CMIP_mask_exp5_baseline_simple")
    df.insert(1, "component", "prediction")
    df.insert(2, "component_order", component_order["prediction"])
    df.insert(3, "scenario", climate)
    df.insert(4, "scenario_order", int(climate_order.index(climate)))
    df.insert(5, "sample_idx", np.arange(len(df)))
    df.insert(6, "latent_dim", int(latent_dim))
    df.insert(7, "prediction_task", prediction_task)
    return df


def _predict_in_batches(model, X, batch_size=4096):
    y_chunks = []
    latent_chunks = []

    model.eval()

    with torch.inference_mode():
        for start in range(0, X.shape[0], batch_size):
            xb = torch.as_tensor(
                X[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )
            y_pred = model(xb)
            z = model.encode(xb)

            y_chunks.append(y_pred.detach().cpu().numpy())
            latent_chunks.append(z.detach().cpu().numpy())

    return np.vstack(y_chunks), np.vstack(latent_chunks)


# Accumulators
meta_dfs_pred = []
truth_arrays_pred, pred_arrays_pred = [], []

n_masked_points = int(len(masked_point_indices))

baseline_predictor.eval()

for climate in climate_order:
    idx = ae_split_indices[climate]["test"]

    X = np.asarray(
        scaled_features_by_climate[climate][idx],
        dtype=np.float32,
    )

    y_true_scaled = np.asarray(
        label_variable_by_climate[climate][idx],
        dtype=np.float32,
    )

    metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    y_hat_scaled, z_np = _predict_in_batches(
        baseline_predictor,
        X,
        batch_size=eval_batch_size,
    )

    y_true_physical = denormalize_masked_targets(
        y_true_scaled,
        variable_means,
        variable_stds,
        n_masked_points=n_masked_points,
        n_variables=n_variables_full,
    )

    y_hat_physical = denormalize_masked_targets(
        y_hat_scaled,
        variable_means,
        variable_stds,
        n_masked_points=n_masked_points,
        n_variables=n_variables_full,
    )

    meta_dfs_pred.append(_build_meta_df(climate, metadata, z_np.shape[1]))
    truth_arrays_pred.append(y_true_physical.astype(np.float32))
    pred_arrays_pred.append(y_hat_physical.astype(np.float32))


baseline_simple_quality_payload = {
    "meta_prediction": pd.concat(meta_dfs_pred, ignore_index=True),
    "truth_prediction": np.concatenate(truth_arrays_pred, axis=0),
    "pred_prediction": np.concatenate(pred_arrays_pred, axis=0),
    "prediction_value_names": masked_prediction_labels,
    "masked_point_indices": masked_point_indices.tolist(),
    "masked_feature_columns": masked_feature_columns.tolist(),
    "visible_point_indices": visible_point_indices.tolist(),
    "visible_feature_columns": visible_feature_columns.tolist(),
}

display(baseline_simple_quality_payload["meta_prediction"].head())
display(baseline_simple_quality_payload["truth_prediction"].shape)


In [ ]:
# Delete the checkpoint now that all evaluation is complete.
if baseline_checkpoint_path.exists():
    baseline_checkpoint_path.unlink()
    print(f"[Checkpoint] {baseline_checkpoint_path.name} deleted.")